# MAPK Base-Editing Screens — Other Figures

The manuscript figures that are not screen figures: competitive growth, CellTiter-Glo viability, inhibitor and kinase dose response, BaF3 IL3-independent growth, and a meta-analysis of published KRAS deep mutational scanning.

Section 0 provisions the environment from scratch, so no pre-existing conda environment is needed. Sections 4 onward are independent of each other and can be run selectively once sections 0–3 have run.

## 0. Environment

Installs the pinned dependency set into the runtime. In Colab, also clones the
repository if the notebook is running standalone.

In [ ]:
#@title Install dependencies { display-mode: "form" }
# Provisions the runtime from scratch on every run, so no persistent environment is required. Takes a few minutes on a cold Colab runtime.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The repository holds the code and TableS7-OtherFiguresData.xlsx.
    # Set REPO_URL to clone from git, or mount Drive and point REPO_DIR at the copy there.
    REPO_DIR = "/content" # @param {type:"string"}
    REPO_DIR = Path(REPO_DIR)
    REPO_URL = "" # @param {type:"string"}

    if REPO_URL and not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    elif not REPO_DIR.exists():
        from google.colab import drive
        drive.mount("/content/drive")
        raise SystemExit(
            f"{REPO_DIR} not found. Set REPO_URL, or set REPO_DIR to the "
            "repository folder inside /content/drive."
        )
    os.chdir(REPO_DIR)

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "TableS1-ScreenData.xlsx").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

# Figure text is Arial; matplotlib resolves it from the working directory first.
if not (REPO_ROOT / "Arial.ttf").exists():
    subprocess.run([
        "wget", "-q",
        "https://raw.githubusercontent.com/liaulab/be-scan/main/be_scan/figure_plot/Arial.ttf",
        "-O", "Arial.ttf",
    ], check=False)

print("Environment ready.")

## 1. Imports

Every import used anywhere in the notebook.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Manuscript-specific code.
sys.path.insert(0, str(Path.cwd()))
# `code` is also a Python standard-library module.
sys.modules.pop("code", None)
from code import config as cfg, other_charts as oc, other_figures as of
print("code/ modules loaded.")

# Chained-assignment and seaborn categorical warnings only.
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 2. Shared configuration

Genes, colors, domain boundaries, condition lists, cutoffs and output paths are defined once in code/config.py and used by every section below.

In [ ]:
#@title Style and palette { display-mode: "form" }

HARMONIZE_A549_A375_COLORS = False # @param {type:"boolean"}

cfg.make_other_figure_dirs()
oc.apply_figure_style()

OUT = cfg.OUT_OTHER_FIGURES
GREY, DARK_GREY, GREEN, BLUE, PURPLE, RED = (
    cfg.OTHER_FIG_COLORS[key]
    for key in ("grey", "dark_grey", "green", "blue", "purple", "red")
)

print(f"Input workbook: {cfg.OTHER_FIGURES_XLSX.name}")
print(f"Output:         {OUT}")
print(f"Jitter seed:    {cfg.OTHER_FIG_JITTER_SEED}")

## 3. Load the input workbook

Prism-style wide sheets melted to long format, and FCS Express exports parsed out of their filenames.

In [ ]:
TABS = {
    "CG_A549": "competitive growth, A549",
    "CG_A375": "competitive growth, A375",
    "CG_KRAS_Y4": "competitive growth, KRAS Y4",
    "CG_MEKi": "competitive growth, MEK inhibitor panel",
    "CG_sgRNA_key": "sgRNA display labels",
    "CG_drug_key": "drug display labels and order",
    "CTG_AZ628_synergy": "CellTiter-Glo, AZ628 x MEKi synergy",
    "CTG_AZ628_BE_clone": "CellTiter-Glo, clone panel +/- AZ628",
    "CTG_clone_d4": "CellTiter-Glo, clone 27 vs NT at day 4",
    "BaF3_JW311": "BaF3 growth, FCS Express counts",
    "BaF3_JW296": "BaF3 growth, volumetric counts",
    "DR_5node": "dose response, five-node inhibitor panel",
    "DR_MEKi": "dose response, MEK inhibitor panel",
    "Kinase_CRAF": "CRAF kinase-domain FRET assay",
    "KRAS_DMS": "published KRAS DMS scores",
}

loaded = pd.DataFrame(
    [{"tab": tab, "contents": description, "rows": len(of.read_tab(tab))}
     for tab, description in TABS.items()])

print(f"{len(loaded)} tabs, {loaded['rows'].sum():,} rows")
loaded

## 4. Competitive growth assays

GFP-labelled edited cells are mixed with unlabelled parental cells. The GFP+ fraction over time reports the relative fitness of the edit under each drug.

In [ ]:
#@title A549 and A375 (JW208) { display-mode: "form" }

A549_A375_GUIDES = ["NT control", "CRAF S257P", "CRAF N392S+E393G"]
A549_A375_LEGEND = {"A549": ["DMSO", "Avutometinib", "Selumetinib"],
                    "A375": ["DMSO", "Selumetinib", "Avutometinib"]}

for cell_line in ["A549", "A375"]:
    df = of.expand_baseline(of.load_competitive_growth(f"CG_{cell_line}")
                            .assign(Experiment=cell_line))
    legend_order = A549_A375_LEGEND["A549" if HARMONIZE_A549_A375_COLORS else cell_line]

    oc.grid_plot(
        df[df["drug_label"].isin(legend_order)],
        groups=A549_A375_GUIDES,
        palette=[GREY, GREEN, BLUE], legend_order=legend_order,
        ncols=3, panel_size=(1.1, 1.5), x_ticks=[0, 5, 12],
        output_path=OUT / f"{cell_line}_comp_growth_Sel_Avut.pdf",
    )

In [ ]:
#@title KRAS Y4C+K5G/E (CF003) { display-mode: "form" }

kras_cg = of.expand_baseline(of.load_competitive_growth("CG_KRAS_Y4"))

oc.grid_plot(
    kras_cg,
    groups=["KRAS Y4C+K5G/E", "Nontargeting(ABE)"],
    palette=[GREY, "#7781BF", "#77cae5"], legend_order=["DMSO", "SHP2i", "KRASi"],
    ncols=2, panel_size=(1.2, 1.83),
    x_ticks=[-1, 0, 2, 4, 6, 8, 10, 12, 14], x_pad=0.25, line_width=0.5,
    output_path=OUT / "KRAS_Y4_NTA_K9.pdf",
);

In [ ]:
#@title MEK inhibitor panel (JW227) { display-mode: "form" }

meki_fc = of.fold_change_vs_nt(of.load_competitive_growth("CG_MEKi"))
meki_fc = meki_fc.rename(columns={"sgRNA_label": "sgRNA_mut_names",
                                  "drug_label": "drug_names"})

MEKI_DRUG_ORDER = ["DMSO", "Avutometinib", "GDC-0623", "Trametiglue", "Trametinib",
                   "Selumetinib", "Mirdametinib", "Cobimetinib", "MAP855", "SHP2i"]
VALIDATED = ["CRAF E393G+N392S", "CRAF G356K", "CRAF G358N", "CRAF N473S+I474V",
             "CRAF L573P", "CRAF K493G", "CRAF S497G", "CRAF S257P",
             "BRAF E501G", "BRAF G466K", "BRAF ~aa467*", "BRAF S365P"]
NON_VALIDATED = ["CRAF_non-editing(aa21)", "CRAF R191K", "CRAF P597F", "CRAF L598F",
                 "BRAF P705L+L706F", "ARAF P558F L559F", "MAP2K1 P232L",
                 "MAP2K1 A391V"]

validated_matrix = oc.plot_heatmap(
    meki_fc, row_order=VALIDATED, col_order=MEKI_DRUG_ORDER, figsize=(3.7, 2.7),
    output_path=OUT / "MEKi_validated_sgRNAs.pdf")

non_validated_matrix = oc.plot_heatmap(
    meki_fc, row_order=NON_VALIDATED, col_order=MEKI_DRUG_ORDER, figsize=(3.4, 2.7),
    output_path=OUT / "MEKi_non_validated_sgRNAs.pdf")

## 5. CellTiter-Glo drug response

384-well viability readouts. Two normalisations are used:

- **`Value_norm`** — divide by the mean of the no-Drug-A wells *within each series*, so every curve starts at 1 and the shapes are directly comparable.
- **`Value_norm_zeroAB`** — divide by the mean of the wells with neither drug, per plate, so the vertical offset between series reflects the Drug B effect.

In [ ]:
#@title AZ628 x MEK inhibitor synergy (JW257) { display-mode: "form" }

synergy = of.read_tab("CTG_AZ628_synergy")
synergy["category"] = np.where(synergy["Drug B nM"] == 0, "DMSO", synergy["Drug B"])
synergy = of.normalize_to_zero_dose(synergy)
synergy = of.normalize_to_zero_both(synergy)
d5 = synergy[synergy["Plate"].astype(str) == "1"]

SYNERGY_ORDER = ["DMSO", "Avutometinib", "Cobimetinib"]
SYNERGY_LABELS = ["DMSO", "Avutometinib [300nM]", "Cobimetinib[300nM]"]
synergy_style = dict(
    x_col="Drug A nM", group_col="category", group_order=SYNERGY_ORDER,
    colors=[GREY, GREEN, BLUE], legend_labels=SYNERGY_LABELS,
    figsize=(3, 1.8), x_scale="linear", x_label="AZ628 [nM]", y_label="cell viability",
    title="Drug A response by category",
    x_ticks=[0, 5, 10, 11], y_ticks=[0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2],
)

oc.plot_mean_sd_curves(d5, y_col="Value_norm_zeroAB", **synergy_style,
                       output_path=OUT / "AZ628_synergy_d5_raw_linear.pdf")

oc.plot_mean_sd_curves(d5, y_col="Value_norm", **synergy_style,
                       output_path=OUT / "AZ628_synergy_d5_0norm_linear.pdf");

In [ ]:
#@title Clone panel +/- AZ628 (JW265) { display-mode: "form" }

be_clone = of.read_tab("CTG_AZ628_BE_clone")
be_clone["category"] = be_clone["ID"]
be_clone = of.normalize_to_zero_dose(be_clone)
az628 = be_clone[be_clone["Plate"].astype(str).str.contains("NT_27")]

AZ628_ORDER = ["NT", "NT+AZ628",
               "Clone 27 (E393G homo, N392S homo)",
               "Clone 27 (E393G homo, N392S homo)+AZ628"]

oc.plot_mean_sd_curves(
    az628, x_col="Drug A nM", y_col="Value_norm", group_col="category",
    group_order=AZ628_ORDER,
    colors=["#C8C6C6", "#707070", "#A2D2FF", "#22577A"],
    legend_labels=["Nontargeting", "Nontargeting + AZ628",
                   "E393G+N392S", "E393G+N392S + AZ628"],
    x_scale="log", figsize=(3.5, 1.8), title="Sensitivity Cobimetinib",
    x_label="Cobimetinib [nM]", y_label="Cell viability",
    y_ticks=[0.2, 0.4, 0.6, 0.8, 1.0, 1.2],
    output_path=OUT / "AZ628_cobimetinib_norm_log.pdf",
);

In [ ]:
#@title Clone 27 vs NT dose response, day 4 (JW262) { display-mode: "form" }

clone_d4 = of.read_tab("CTG_clone_d4")
clone_d4["category"] = clone_d4["Plate"] + " " + clone_d4["ID"]
clone_d4 = of.normalize_to_zero_dose(clone_d4)
clone_d4_tidy = clone_d4[clone_d4["Drug A nM"] > 0]

CLONE_LABELS = {f"d4_{drug} {clone}": label
                for drug in ("Cobimetinib", "Avutometinib")
                for clone, label in [("NT", "NT sgRNA clone"),
                                     ("Clone 27", "CRAF N392S+E393G")]}
CLONE_TICKS = [(10 ** e, rf"$10^{{{e}}}$") for e in range(0, 5)]

clone_fits = {}
for drug in ["Cobimetinib", "Avutometinib"]:
    _, _, clone_fits[drug] = oc.plot_dose_response(
        clone_d4_tidy, conc_col="Drug A nM", response_col="Value_norm",
        group_col="category",
        group_order=[f"d4_{drug} NT", f"d4_{drug} Clone 27"],
        colors=[GREY, PURPLE], label_map=CLONE_LABELS,
        x_label="Drug [nM]", y_label="Normalized viability", title=drug,
        x_tick_labels=CLONE_TICKS, figsize=(2.75, 1.5), point_size=8,
        output_path=OUT / f"clone_d4_{drug[:4].lower()}_dose_response_narrow.pdf",
    )

clone_ec = pd.concat([of.ecx_table(fits).assign(drug=drug)
                      for drug, fits in clone_fits.items()], ignore_index=True)
clone_ec.to_csv(OUT / "clone-d4-EC-values.csv", index=False)
clone_ec[["drug", "group", "EC50", "EC90", "EC10", "bottom", "top"]]

## 6. Inhibitor and kinase dose–response curves

Logistic fit curves on a log concentration axis.

In [ ]:
DECADE_TICKS = [(10 ** e, rf"$10^{{{e}}}$") for e in range(-1, 6)]

In [ ]:
#@title Five-node inhibitor panel { display-mode: "form" }

five_node = of.load_dose_response("DR_5node")

_, _, five_node_fits = oc.plot_dose_response(
    five_node, conc_col="dose_nM", response_col="response", group_col="condition",
    group_order=["Batoprotafib (JW130)", "Adagrasib (JW130)", "LY3009120 (JW138)",
                 "Trametinib (JW138)", "Temuterkib (JW130)"],
    colors=[PURPLE, BLUE, "#E7A075", GREEN, RED],
    label_map={"Batoprotafib (JW130)": "Batoprotafib (SHP2i)",
               "Adagrasib (JW130)": "Adagrasib (KRASi)",
               "LY3009120 (JW138)": "LY3009120 (RAFi)",
               "Trametinib (JW138)": "Trametinib (MEKi)",
               "Temuterkib (JW130)": "Temuterkib (ERKi)"},
    x_label="Inibitor [nM]", y_label="%viability", title="",
    x_tick_labels=DECADE_TICKS,
    y_tick_labels=[(v, str(v)) for v in range(0, 141, 20)],
    figsize=(3.25, 2), point_size=8,
    output_path=OUT / "5_node_growth_curves.pdf",
)

five_node_ec = of.ecx_table(five_node_fits)
five_node_ec.to_csv(OUT / "5-node-EC-values.csv", index=False)
five_node_ec[["group", "EC50", "EC90", "EC10", "bottom", "top", "hill_slope"]]

In [ ]:
#@title MEK inhibitor panel { display-mode: "form" }

meki_dr = of.load_dose_response("DR_MEKi")
meki_dr["response"] = meki_dr["response"] * 100

MEKI_DR_ORDER = ["Avutametinib (JW138)", "GDC-0623 (JW138)", "Trametiglue (JW138)",
                 "Trametinib (JW138)", "Selumetinib (JW197)", "Mirdametinib (JW197)",
                 "Cobimetinib (JW204)", "MAP855 (JW197)"]

_, _, meki_dr_fits = oc.plot_dose_response(
    meki_dr, conc_col="dose_nM", response_col="response", group_col="condition",
    group_order=MEKI_DR_ORDER,
    colors=["#1f77b4", "#ffbb78", "#98df8a", "#9edae5",
            "#c5b0d5", "#f7b6d2", "#c7c7c7", "#ff9896"],
    label_map={g: g.split(" (")[0] for g in MEKI_DR_ORDER},
    x_label="Inibitor [nM]", y_label="%viability", title="",
    x_tick_labels=DECADE_TICKS,
    y_tick_labels=[(v, str(v)) for v in range(0, 121, 20)],
    figsize=(3.25, 2), point_size=8,
    output_path=OUT / "MEKi_growth_curves.pdf",
)

meki_dr_ec = of.ecx_table(meki_dr_fits)
meki_dr_ec.to_csv(OUT / "MEKi-dose-response-EC-values.csv", index=False)
meki_dr_ec[["group", "EC50", "EC90", "EC10", "bottom", "top", "hill_slope"]]

In [ ]:
#@title CRAF kinase-domain FRET assay { display-mode: "form" }

kinase = of.load_dose_response("Kinase_CRAF")

KINASE_ORDER = ["CRAFKD,SSDD  (3/12 result)", "CRAFKD,SSDD,E393G (3/12 result)",
                "CRAFKD,SSDD,N392S/E393G (3/12 result)",
                "CRAFKD,SSDD,G358N (3/12 result)",
                "CRAFKD,SSDD,N473S/I474V (3/12 result)"]

oc.plot_dose_response(
    kinase, conc_col="dose_nM", response_col="response", group_col="condition",
    group_order=KINASE_ORDER,
    colors=[DARK_GREY, PURPLE, "#DFCCDE", GREEN, RED],
    label_map=dict(zip(KINASE_ORDER,
                       ["WT", "E393G", "N392S/E393G", "G358N", "N473S/I474V"])),
    x_label="Enzyme [nM]", y_label="FRET Ratio", title="",
    figsize=(3, 2), point_size=8,
    output_path=OUT / "CRAF_3_12_only.pdf",
);

## 7. BaF3 IL3-independent growth

In [ ]:
#@title JW311, gate-corrected counts { display-mode: "form" }

baf3 = of.load_baf3_jw311()

baf3_style = dict(
    x_col="Day", y_col="Cells/ml", group_col="label", scale_factor=1e6, sci_y=True,
    figsize=(1.5, 1.5), title="Total Cells Over Time (FCS Express)",
    x_label="Day", y_label=r"Cells $\times$ 10$^{6}$/ ml",
    x_lim=(-0.5, 14), x_ticks=(0, 2, 4, 6, 8, 10, 12, 14),
    point_alpha=0.35, marker_size=3, line_width=0.5, elinewidth=None,
    cap_thick=0.5, spine_width=0.5, tick_length=4,
    legend_loc="upper left", legend_anchor=(1.02, 1),
    layout="reserve", transparent=True,
)

PALE = ["#606161", "#aec7e8", "#ffbb78", "#98df8a", "#c5b0d5",
        "#ff9896", "#c49c94", "#f7b6d2", "#dbdb8d"]
BE_COLORS = [GREY, DARK_GREY, RED, BLUE]

print(f"{len(baf3):,} wells, {baf3['label'].nunique()} constructs")

wt_singles = ["WT", "Y4C", "Y4A", "Y4E", "Y4F", "K5E", "K5G", "K5A", "K5N"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, wt_singles), group_order=wt_singles, colors=PALE, **baf3_style,
    y_lim=(0, 0.8), y_ticks=(-0.05, 0, 0.5, 0.8),
    output_path=OUT / "BaF3_KRAS_strong_WT_background_singles_only.pdf")

g12c_singles = ["G12C", "Y4A+G12C", "Y4E+G12C", "Y4F+G12C",
                "K5E+G12C", "K5G+G12C", "K5A+G12C", "K5N+G12C"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, g12c_singles), group_order=g12c_singles,
    colors=["#606161", "#ffbb78", "#98df8a", "#c5b0d5",
            "#ff9896", "#c49c94", "#f7b6d2", "#dbdb8d"], **baf3_style,
    y_lim=(0, 2.5), y_ticks=(-0.05, 0, 0.5, 1, 1.5, 2, 2.5),
    output_path=OUT / "BaF3_KRAS_strong_G12C_background_singles_only.pdf")

wt_be = ["no IL3", "WT", "Y4C+K5E", "Y4C+K5G"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, wt_be, include_no_il3=True), group_order=wt_be,
    colors=BE_COLORS, **baf3_style, y_lim=(0, 0.2), y_ticks=(-0.02, 0, 0.1, 0.2),
    output_path=OUT / "BaF3_KRAS_strong_WT_background_BE_edits_only.pdf")

g12c_be = ["no IL3", "G12C", "Y4C+K5E+G12C", "Y4C+K5G+G12C"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, g12c_be, include_no_il3=True), group_order=g12c_be,
    colors=BE_COLORS, **baf3_style, y_lim=(0, 2.5),
    y_ticks=(-0.05, 0, 0.5, 1, 1.5, 2, 2.5),
    output_path=OUT / "BaF3_KRAS_strong_G12C_background_BE_edits_only.pdf")

q61 = ["WT", "Q61R", "Y4C+K5G+Q61R", "G12C", "Y4C+K5G+G12C"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, q61), group_order=q61,
    colors=[GREY, "#9edae5", "#17becf", DARK_GREY, RED], **baf3_style,
    y_lim=(0, 2.5), y_ticks=(-0.05, 0, 0.5, 1, 1.5, 2, 2.5),
    output_path=OUT / "BaF3_KRAS_strong_Q61R.pdf")

weak_singles = ["G12C", "Y4C+G12C", "Y4A+G12C", "Y4E+G12C",
                "Y4F+G12C", "K5E+G12C", "K5G+G12C", "K5A+G12C"]
oc.plot_mean_sd_curves(
    of.baf3_subset(baf3, weak_singles, promoter="Weak"), group_order=weak_singles,
    colors=PALE[:8], **baf3_style, y_lim=(0, 0.8), y_ticks=(-0.05, 0, 0.5, 0.8),
    output_path=OUT / "BaF3_KRAS_weak_G12C_background_singles_only.pdf")

NRAS_COLORS = [DARK_GREY, RED, "#98df8a", "#c5b0d5"]
for background, labels in [("WT", ["WT", "Y4C+K5G", "Y4E", "Y4F"]),
                           ("G12C", ["G12C", "Y4C+K5G+G12C", "Y4E+G12C", "Y4F+G12C"])]:
    oc.plot_mean_sd_curves(
        baf3[(baf3["Gene"] == "NRAS") & (baf3["label"].isin(labels))],
        group_order=labels, colors=NRAS_COLORS, **baf3_style,
        y_lim=(0, 1.5), y_ticks=(-0.05, 0, 0.5, 1, 1.5),
        output_path=OUT / f"BaF3_NRAS_strong_{background}_background.pdf")

In [ ]:
#@title JW296, weak-promoter KRAS variants { display-mode: "form" }

baf3_jw296 = of.read_tab("BaF3_JW296")

jw296_style = dict(
    x_col="Day", y_col="Cells/ml", group_col="Construct", colors=BE_COLORS,
    scale_factor=1.0, sci_y=True, figsize=(1.5, 1.5), title="",
    x_label="Day", y_label="Total Cells",
    x_lim=(-0.5, 14), x_ticks=(0, 2, 4, 6, 8, 10, 12, 14),
    point_alpha=0.35, marker_size=3, line_width=1.0, elinewidth=None,
    cap_thick=0.5, spine_width=0.8, tick_length=4,
    legend_loc="upper left", legend_anchor=(1.02, 1),
    layout="reserve", transparent=True,
)

for background, labels, y_lim, y_ticks in [
    ("WT", ["NIC -IL3", "WT", "Y4C, K5G", "Y4C, K5E"],
     (-4e4, 3e5), (-2e4, 0, 1e5, 2e5, 3e5)),
    ("G12C", ["NIC -IL3", "G12C", "G12C, Y4C, K5G", "G12C, Y4C, K5E"],
     (-4e4, 2e6), (-25e4, 0, 1e6, 2e6)),
]:
    oc.plot_mean_sd_curves(
        baf3_jw296[baf3_jw296["Construct"].isin(labels)],
        group_order=labels, **jw296_style, y_lim=y_lim, y_ticks=y_ticks,
        output_path=OUT / f"BaF3_JW296_total_cells_BE_KRAS_variants_{background}_background.pdf")

## 8. Published KRAS DMS meta-analysis

Deep mutational scanning scores from several published screens. Each assay column is z-scored across all variants. Then the substitutions at Q61, Y4 and K5 are shown against that distribution.

In [ ]:
#@title Site boxplots { display-mode: "form" }

dms = of.read_tab("KRAS_DMS")
dms_z = of.zscore_columns(dms)

DMS_CONDITIONS = ["HA1E_WT_LFC", "Lito_BI2865vsVEH_N2023", "Lito_RMC4998vsVEH_S2023",
                  "Aguirre_MRTX1257_T0", "Aguirre_Sotorasib_T0"]
DMS_LABELS = {"HA1E_WT_LFC": "Kwon HA1E Transformation",
              "Lito_BI2865vsVEH_N2023": "Kim BI2865 Resistance",
              "Lito_RMC4998vsVEH_S2023": "Schulze RMC4998 Resistance",
              "Aguirre_MRTX1257_T0": "Awad MRTX1257",
              "Aguirre_Sotorasib_T0": "Awad Sotorasib"}

print(f"{len(dms_z):,} variants across {len(DMS_CONDITIONS)} published assays")

oc.plot_site_boxplots(
    dms_z[["mutation_id"] + DMS_CONDITIONS],
    sites=["Q61", "Y4", "K5"], highlight_muts=["Q61L", "Y4C", "K5G"],
    label_map=DMS_LABELS, figsize=(1, 2.7), title="",
    output_path=OUT / "KRAS_DMS_Q61_Y4_K5_stacked_boxplots.pdf",
);

## 9. Run summary

Everything written by this notebook.

In [ ]:
written = sorted(p for p in OUT.rglob("*") if p.is_file())
by_type = pd.Series([p.suffix for p in written])

print(f"{len(written)} files written to {OUT}\n")
print(by_type.value_counts().rename("files").to_string())
print()
for path in written:
    print(f"  {path.name}")